# 04 — Robustness checks and extensions

**Part 4.** This notebook stress-tests the stated-versus-revealed preference story and adds event-context and participant-segment analyses.

## Questions

1. Do other participants' ratings of the same date explain the focal participant's choice, or does their *own rating relative to peer consensus* still matter? (consensus / halo sensitivity)
2. Is yes propensity different in the limited-choice versus extensive-choice condition, and does it change over the sequence of dates? (choice architecture and order/fatigue)
3. How much do participants' stated preference weights differ from their retrospective account of what mattered during the event? (pre-event vs follow-up self-report)
4. Can participants be grouped by the size of the gap between stated weights and a simple person-level behavioral preference proxy?

## Read this before interpreting results

- Run `01_cleaning.ipynb` first. This notebook reads its `data/processed/clean_unscaled.csv` output; it does not depend on the repository's unprovided `src/` modules.
- In the data key, `condtn=1` means limited choice and `condtn=2` means extensive choice; `order` is the date number that night. `attr7_2` is a next-day retrospective report asking participants to recall which attributes mattered in their decisions. That is a change in *reported weights*, not proof of learning or a prospective preference update.
- The date ratings are subjective and collected after each interaction. The peer-rating sensitivity is therefore a consensus control, not an objective quality measure or a causal “halo effect” estimate.
- The person-level behavioral weights in the final section are an exploratory index from each participant's yes/no choices and post-date ratings. It is not the same estimator as the gender-level conditional logit in `02_core_model.ipynb` and should not be described as a stable individual preference or as causal.

**Marketing link:** these extensions connect preference measurement to recommendation quality, choice architecture, and audience segmentation. The data do not contain subscription prices, spending, retention, or lifetime revenue, so they cannot support CLV or pricing claims.

**Variable reference:** [Speed Dating Data Key](https://web.mit.edu/17.871/www/2009/speed_dating_codebook.pdf).

In [1]:
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import chi2, norm

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, 'to_string') else value)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 150)

ATTRS = ['attr', 'sinc', 'intel', 'fun', 'amb', 'shar']
PREF_PREFIXES = ['attr', 'sinc', 'intel', 'fun', 'amb', 'shar']
STATED = [f'{p}1_1' for p in PREF_PREFIXES]
FOLLOWUP = [f'{p}7_2' for p in PREF_PREFIXES]
SEED = 42
N_BOOT = 1000

# Set this to a path string if automatic discovery does not find the cleaned CSV.
DATA_PATH = None
if DATA_PATH is None:
    roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    candidates = []
    for root in roots:
        candidates.extend([
            root / 'data' / 'processed' / 'clean_unscaled.csv',
            root / 'processed' / 'clean_unscaled.csv',
            root / 'upload' / 'Speed Dating cleaned unscaled.csv',
            root / 'upload' / 'clean_unscaled.csv',
        ])
    DATA_PATH = next((p for p in candidates if p.is_file()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        'Could not find data/processed/clean_unscaled.csv. Run 01_cleaning.ipynb, '
        'or set DATA_PATH above to the cleaned CSV location.'
    )

DATA_PATH = Path(DATA_PATH)
data = pd.read_csv(DATA_PATH)
required = ['iid', 'pid', 'wave', 'gender', 'dec', *ATTRS]
missing = [c for c in required if c not in data.columns]
if missing:
    raise ValueError(f'The cleaned CSV is missing required columns: {missing}')

for col in ['iid', 'pid', 'wave', 'gender', 'dec', 'condtn', 'order', 'match', *ATTRS, *STATED, *FOLLOWUP]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')
data = data.loc[data['dec'].isin([0, 1])].copy()

print(f'Source: {DATA_PATH}')
print(f'{len(data):,} date rows; {data.iid.nunique():,} participants; {data.wave.nunique():,} waves')
print(f'Observed yes rate: {data.dec.mean():.3f}')

Source: /workspace/scratch/7308692dcc46/upload/Speed Dating cleaned unscaled.csv
8,378 date rows; 551 participants; 21 waves
Observed yes rate: 0.420


## 1. Sample support and data coverage

First check that the fields needed by each extension are present and that condition is constant within participant, as expected for a between-person event condition. The wave-by-condition table tells us whether we can compare condition groups *within the same wave*; without overlap, condition is confounded with event wave in this dataset.

In [2]:
coverage = pd.DataFrame({
    'field': ['condtn', 'order', 'match', *STATED, *FOLLOWUP],
    'available': [c in data.columns for c in ['condtn', 'order', 'match', *STATED, *FOLLOWUP]],
})
display(coverage)

if 'condtn' in data.columns:
    condtn_per_person = data.groupby('iid')['condtn'].nunique(dropna=True)
    print(f"Participants with >1 recorded condition value: {(condtn_per_person > 1).sum()}")
    person_condition = data.groupby('iid', as_index=False).agg(
        gender=('gender', 'first'), wave=('wave', 'first'), condtn=('condtn', 'first'),
        n_dates=('dec', 'size'), yes_rate=('dec', 'mean')
    )
    if 'match' in data.columns:
        person_match = data.groupby('iid')['match'].mean().rename('mutual_match_rate')
        person_condition = person_condition.merge(person_match, on='iid', how='left')
    display(person_condition.groupby(['condtn', 'gender'], dropna=False).agg(
        participants=('iid', 'nunique'), median_dates=('n_dates', 'median'),
        mean_yes_rate=('yes_rate', 'mean'),
        **({'mean_mutual_match_rate': ('mutual_match_rate', 'mean')} if 'mutual_match_rate' in person_condition.columns else {})
    ).round(3))

    condition_by_wave = pd.crosstab(person_condition['wave'], person_condition['condtn'])
    display(condition_by_wave)
    condition_levels = set(person_condition['condtn'].dropna().unique())
    overlap_waves = condition_by_wave.index[condition_by_wave.reindex(columns=sorted(condition_levels), fill_value=0).gt(0).all(axis=1)].tolist() if len(condition_levels) > 1 else []
    print(f'Waves containing both condition levels: {overlap_waves}')

       field  available
0     condtn       True
1      order       True
2      match       True
3    attr1_1       True
4    sinc1_1       True
5   intel1_1       True
6     fun1_1       True
7     amb1_1       True
8    shar1_1       True
9    attr7_2       True
10   sinc7_2       True
11  intel7_2       True
12    fun7_2       True
13    amb7_2       True
14   shar7_2       True
Participants with >1 recorded condition value: 0
               participants  median_dates  mean_yes_rate  mean_mutual_match_rate
condtn gender                                                                   
1      0                 81           9.0          0.431                   0.198
       1                 84          10.0          0.493                   0.196
2      0                193          18.0          0.350                   0.156
       1                193          19.0          0.476                   0.157
condtn   1   2
wave          
1       20   0
2        0  35
3       20   0
4     

## 2. Consensus / halo sensitivity

For each date row `(iid, pid, wave)`, construct the mean rating that **other participants in the same wave** gave the focal participant's date partner. The focal row is left out of that average. This direction matters: we aggregate rows where the partner is the rated person; the dataset's `attr_o`-style fields record the opposite direction.

We then decompose the focal rating into (a) peer consensus and (b) the focal person's deviation from that consensus. Both are put on the same within-gender scale. The baseline is the core-style conditional logit on the focal person's six ratings. The augmented model lets consensus and the person's own deviation have separate coefficients. A likelihood-ratio test asks whether the split adds explanatory information. A persistent deviation coefficient suggests choices contain idiosyncratic signal beyond consensus; it does not prove that consensus creates a halo.

In [3]:
def informative_rows(frame, predictors):
    d = frame.dropna(subset=['iid', 'dec', *predictors]).copy()
    informative_iids = d.groupby('iid')['dec'].nunique()
    informative_iids = informative_iids[informative_iids.eq(2)].index
    return d[d['iid'].isin(informative_iids)].copy()


def _partition_logz_and_mean(beta, X, k):
    """Compute log elementary-symmetric partition function and E[sum(x)]."""
    eta = X @ beta
    shift = float(np.max(eta))
    weights = np.exp(eta - shift)
    p = X.shape[1]
    z = np.zeros(k + 1, dtype=float)
    first = np.zeros((k + 1, p), dtype=float)
    z[0] = 1.0
    seen = 0
    for row, weight in zip(X, weights):
        upper = min(k, seen + 1)
        for j in range(upper, 0, -1):
            z_prev = z[j - 1]
            first_prev = first[j - 1].copy()
            z[j] += weight * z_prev
            first[j] += weight * (first_prev + z_prev * row)
        seen += 1
    if z[k] <= 0 or not np.isfinite(z[k]):
        raise FloatingPointError('Conditional-logit partition calculation failed.')
    log_z = np.log(z[k]) + k * shift
    expected_sum_x = first[k] / z[k]
    return log_z, expected_sum_x


def fit_conditional_logit(frame, predictors):
    """Exact conditional logit, conditioning on each participant's yes count."""
    d = informative_rows(frame, predictors)
    if d['iid'].nunique() < 2:
        raise ValueError('Too few participants with both yes and no outcomes for conditional logit.')
    groups = []
    for _, group in d.groupby('iid', sort=False):
        Xg = group[predictors].to_numpy(dtype=float)
        yg = group['dec'].to_numpy(dtype=int)
        k = int(yg.sum())
        if 0 < k < len(yg):
            groups.append((Xg, Xg[yg == 1].sum(axis=0), k))

    def objective(beta):
        nll = 0.0
        gradient = np.zeros(len(predictors), dtype=float)
        for Xg, observed_sum_x, k in groups:
            log_z, expected_sum_x = _partition_logz_and_mean(beta, Xg, k)
            nll += log_z - beta @ observed_sum_x
            gradient += expected_sum_x - observed_sum_x
        return nll, gradient

    opt = minimize(objective, np.zeros(len(predictors)), jac=True, method='BFGS',
                   options={'maxiter': 300, 'gtol': 1e-7})
    if not opt.success and np.linalg.norm(opt.jac) > 1e-4:
        print(f'Warning: conditional-logit optimizer status: {opt.message}; gradient norm={np.linalg.norm(opt.jac):.3g}')
    result = SimpleNamespace(
        params=pd.Series(opt.x, index=predictors), llf=-float(opt.fun),
        success=bool(opt.success), message=str(opt.message)
    )
    return result, d

if 'pid' not in data.columns or 'wave' not in data.columns:
    raise ValueError('The consensus analysis requires pid and wave.')

halo = data.dropna(subset=['iid', 'pid', 'wave', 'gender', 'dec']).copy()
for attr in ATTRS:
    keys = halo.groupby(['wave', 'pid'])[attr]
    total = keys.transform('sum')
    n_raters = keys.transform('count')
    focal_has_rating = halo[attr].notna().astype(int)
    n_other_raters = n_raters - focal_has_rating
    peer_total = total - halo[attr].fillna(0)
    halo[f'peer_{attr}'] = (peer_total / n_other_raters.where(n_other_raters > 0)).where(n_other_raters > 0)
    halo[f'deviation_{attr}'] = halo[attr] - halo[f'peer_{attr}']

peer_cols = [f'peer_{a}' for a in ATTRS]
dev_cols = [f'deviation_{a}' for a in ATTRS]
halo = halo.dropna(subset=[*ATTRS, *peer_cols, *STATED]).copy()
halo = informative_rows(halo, [*ATTRS, *peer_cols])

halo_rows = []
for gender_value, gender_label in [(0, 'Women'), (1, 'Men')]:
    g = halo.loc[halo['gender'].eq(gender_value)].copy()
    if g.empty:
        continue
    own_z, peer_z, dev_z = [], [], []
    for a in ATTRS:
        mean = g[a].mean()
        sd = g[a].std(ddof=0)
        if not np.isfinite(sd) or sd == 0:
            raise ValueError(f'Cannot standardize {a} ratings for {gender_label}.')
        own_name, peer_name, dev_name = f'own_{a}_z', f'peer_{a}_z', f'dev_{a}_z'
        g[own_name] = (g[a] - mean) / sd
        g[peer_name] = (g[f'peer_{a}'] - mean) / sd
        # Same centering/scaling as the own rating, so own_z == peer_z + deviation_z.
        g[dev_name] = g[own_name] - g[peer_name]
        own_z.append(own_name)
        peer_z.append(peer_name)
        dev_z.append(dev_name)

    baseline, db = fit_conditional_logit(g, own_z)
    split, ds = fit_conditional_logit(g, [*peer_z, *dev_z])
    lr = max(0.0, 2 * (split.llf - baseline.llf))
    df_lr = len(split.params) - len(baseline.params)
    p_lr = chi2.sf(lr, df_lr) if df_lr > 0 else np.nan
    halo_rows.append({
        'gender': gender_label, 'n_dates': len(ds), 'n_people': ds.iid.nunique(),
        'baseline_llf': baseline.llf, 'split_model_llf': split.llf,
        'LR_stat': lr, 'LR_df': df_lr,
        'LR_p': '<0.001' if p_lr < 0.001 else f'{p_lr:.4f}',
    })
    for a, own_name, peer_name, dev_name in zip(ATTRS, own_z, peer_z, dev_z):
        halo_rows.append({
            'gender': gender_label, 'attribute': a,
            'baseline_own_beta': baseline.params[own_name],
            'peer_consensus_beta': split.params[peer_name],
            'own_minus_peer_beta': split.params[dev_name],
        })

display(pd.DataFrame(halo_rows).round(4))

   gender  n_dates  n_people  baseline_llf  split_model_llf  LR_stat  LR_df    LR_p attribute  baseline_own_beta  peer_consensus_beta  own_minus_peer_beta
0   Women   3000.0     223.0     -719.4246        -698.1942  42.4608    6.0  <0.001       NaN                NaN                  NaN                  NaN
1   Women      NaN       NaN           NaN              NaN      NaN    NaN     NaN      attr             1.6017               1.9787               1.2989
2   Women      NaN       NaN           NaN              NaN      NaN    NaN     NaN      sinc             0.3882               0.2523               0.4418
3   Women      NaN       NaN           NaN              NaN      NaN    NaN     NaN     intel             0.3965               0.6109               0.4401
4   Women      NaN       NaN           NaN              NaN      NaN    NaN     NaN       fun             0.8379               0.6157               0.7861
5   Women      NaN       NaN           NaN              NaN      NaN  

### How to read this check

- `peer_consensus_beta` is the association between other daters' average rating of the partner and the participant's yes decision, conditional on the participant's own rating.
- `own_minus_peer_beta` captures the participant's rating relative to the other daters' average, conditional on consensus.
- The likelihood-ratio comparison is a model diagnostic. People rate multiple dates and partners are rated by multiple people, so the conditional-logit standard errors do not account for every cross-person dependency. Treat p-values as exploratory and emphasize coefficient direction and size.

## 3. Choice condition and choice overload

`condtn` is a between-participant event condition, so it cannot be identified in a participant fixed-effects model. We estimate an exploratory row-level logistic model with standard errors clustered by participant. If both conditions occur within the same waves, the primary comparison uses only those overlap waves and includes wave fixed effects. If there is no within-wave overlap, the notebook reports a gender-adjusted association without wave fixed effects and flags the condition/wave confounding.

The condition contrast is not automatically causal: the codebook labels limited versus extensive choice, but the data do not guarantee that condition assignment is independent of wave or every other event difference.

In [4]:
def fit_clustered_logit(frame, include_wave=False):
    """Logit MLE with participant-clustered sandwich standard errors."""
    d = frame.copy()
    Xdf = pd.DataFrame({
        'intercept': 1.0,
        'extensive_choice': d['condtn'].eq(2).astype(float),
        'male': d['gender'].eq(1).astype(float),
    }, index=d.index)
    if include_wave:
        wave_dummies = pd.get_dummies(d['wave'].astype('category'), prefix='wave', drop_first=True, dtype=float)
        Xdf = pd.concat([Xdf, wave_dummies], axis=1)
    X = Xdf.to_numpy(dtype=float)
    y = d['dec'].to_numpy(dtype=float)

    def objective(beta):
        eta = X @ beta
        prob = expit(eta)
        loss = np.logaddexp(0, eta).sum() - y @ eta
        gradient = X.T @ (prob - y)
        return loss, gradient

    opt = minimize(objective, np.zeros(X.shape[1]), jac=True, method='BFGS',
                   options={'maxiter': 500, 'gtol': 1e-6})
    beta = opt.x
    prob = expit(X @ beta)
    w = prob * (1 - prob)
    bread = np.linalg.pinv(X.T @ (X * w[:, None]))
    scores = X * (y - prob)[:, None]
    cluster_codes, cluster_ids = pd.factorize(d['iid'], sort=False)
    cluster_scores = np.zeros((len(cluster_ids), X.shape[1]), dtype=float)
    np.add.at(cluster_scores, cluster_codes, scores)
    meat = cluster_scores.T @ cluster_scores
    n, p = X.shape
    g = len(cluster_ids)
    correction = (g / (g - 1)) * ((n - 1) / (n - p)) if g > 1 and n > p else 1.0
    covariance = bread @ meat @ bread * correction
    se = np.sqrt(np.maximum(np.diag(covariance), 0))
    z = beta / np.where(se > 0, se, np.nan)
    pvalues = 2 * norm.sf(np.abs(z))
    return SimpleNamespace(
        params=pd.Series(beta, index=Xdf.columns), bse=pd.Series(se, index=Xdf.columns),
        pvalues=pd.Series(pvalues, index=Xdf.columns), converged=bool(opt.success),
        nobs=n, n_people=g
    )

if 'condtn' not in data.columns:
    print('condtn is absent; skip condition model.')
else:
    cond = data.dropna(subset=['iid', 'dec', 'gender', 'condtn', 'wave']).copy()
    cond['condtn'] = cond['condtn'].astype(int)
    levels = sorted(cond['condtn'].unique())
    print(f'Condition values present: {levels} (data key: 1=limited, 2=extensive)')

    if len(levels) == 2 and overlap_waves:
        cond_fit = cond.loc[cond['wave'].isin(overlap_waves)].copy()
        include_wave = True
        condition_note = 'Wave-adjusted estimate on waves with both conditions.'
    else:
        cond_fit = cond
        include_wave = False
        condition_note = 'No within-wave condition comparison available; condition is potentially confounded with wave.'

    condition_result = fit_clustered_logit(cond_fit, include_wave=include_wave)
    print(condition_note)
    print(f'Rows={condition_result.nobs:,}; participants={condition_result.n_people:,}; converged={condition_result.converged}')

    term = 'extensive_choice'
    beta = condition_result.params[term]
    se = condition_result.bse[term]
    ci_low, ci_high = beta - 1.96 * se, beta + 1.96 * se
    condition_result_table = pd.DataFrame([{
        'contrast': 'extensive vs limited (reference condition 1)',
        'log_odds': beta, 'odds_ratio': np.exp(beta),
        'OR_95CI_low': np.exp(ci_low), 'OR_95CI_high': np.exp(ci_high),
        'p_clustered_by_iid': condition_result.pvalues[term],
    }])
    display(condition_result_table.round(4))

Condition values present: [np.int64(1), np.int64(2)] (data key: 1=limited, 2=extensive)
No within-wave condition comparison available; condition is potentially confounded with wave.
Rows=8,378; participants=551; converged=True
                                       contrast  log_odds  odds_ratio  OR_95CI_low  OR_95CI_high  p_clustered_by_iid
0  extensive vs limited (reference condition 1)    -0.199      0.8196       0.6708        1.0013              0.0515


## 4. Order and possible fatigue

`order` is the sequence position of each date that night. To make effect size comparable across participants with different numbers of dates, rescale each person's observed range to 0 (first) through 1 (last). Fit conditional logits with participant fixed effects, first with the six rating controls, then with a linear order term, then with a quadratic term. This asks whether yes propensity changes over the event and whether a straight line is adequate.

An order association is not necessarily fatigue: date quality may vary by order, and the study's ordering process and event context should be considered when interpreting it.

In [5]:
if 'order' not in data.columns:
    print('order is absent; skip order analysis.')
else:
    order_data = data.dropna(subset=['iid', 'gender', 'dec', 'order', *ATTRS, *STATED]).copy()
    order_min = order_data.groupby('iid')['order'].transform('min')
    order_max = order_data.groupby('iid')['order'].transform('max')
    order_data['order_rel'] = (order_data['order'] - order_min) / (order_max - order_min).replace(0, np.nan)
    order_data['order_rel_sq'] = order_data['order_rel'] ** 2
    order_data = order_data.dropna(subset=['order_rel', 'order_rel_sq']).copy()
    order_data = informative_rows(order_data, [*ATTRS, 'order_rel', 'order_rel_sq'])

    order_rows = []
    for gender_value, gender_label in [(0, 'Women'), (1, 'Men')]:
        g = order_data.loc[order_data['gender'].eq(gender_value)].copy()
        if g.empty:
            continue
        rating_z = []
        for a in ATTRS:
            sd = g[a].std(ddof=0)
            if not np.isfinite(sd) or sd == 0:
                continue
            z = f'{a}_z'
            g[z] = (g[a] - g[a].mean()) / sd
            rating_z.append(z)
        if len(rating_z) != len(ATTRS):
            print(f'Skipping {gender_label}: at least one rating has zero variance.')
            continue

        base, d_base = fit_conditional_logit(g, rating_z)
        linear, d_linear = fit_conditional_logit(g, [*rating_z, 'order_rel'])
        quadratic, d_quad = fit_conditional_logit(g, [*rating_z, 'order_rel', 'order_rel_sq'])

        lr_order = max(0.0, 2 * (linear.llf - base.llf))
        p_order = chi2.sf(lr_order, 1)
        lr_curve = max(0.0, 2 * (quadratic.llf - linear.llf))
        p_curve = chi2.sf(lr_curve, 1)
        b_linear = linear.params['order_rel']
        b1 = quadratic.params['order_rel']
        b2 = quadratic.params['order_rel_sq']
        order_rows.append({
            'gender': gender_label, 'n_dates': len(d_quad), 'n_people': d_quad.iid.nunique(),
            'linear_order_beta': b_linear, 'linear_OR_first_to_last': np.exp(b_linear),
            'LR_add_order': lr_order, 'p_add_order': p_order,
            'quadratic_beta': b2, 'quadratic_OR_first_to_last': np.exp(b1 + b2),
            'LR_add_curvature': lr_curve, 'p_add_curvature': p_curve,
        })
    display(pd.DataFrame(order_rows).round(4))

  gender  n_dates  n_people  linear_order_beta  linear_OR_first_to_last  LR_add_order  p_add_order  quadratic_beta  quadratic_OR_first_to_last  LR_add_curvature  p_add_curvature
0  Women     3000       223             0.0504                   1.0516        0.0784       0.7795          0.1525                      1.0532            0.0553           0.8141
1    Men     3222       229             0.0297                   1.0302        0.0290       0.8649          0.6353                      1.0209            1.0299           0.3102


### Interpretation guide

- `linear_OR_first_to_last < 1` means lower yes odds toward the last observed date in the linear specification.
- `p_add_curvature` tests whether the quadratic term improves fit beyond the linear order effect. It is a shape check, not a test of psychological fatigue by itself.
- These are conditional-logit likelihood comparisons; the same participant is the unit of conditioning, but repeated partner/event dependence is not fully modeled.

## 5. Pre-event stated preferences versus next-day recollection

The follow-up `attr7_2` family asks participants to retrospectively allocate importance to the attributes that mattered in their event decisions. Compare it with the initial `*1_1` preference allocation using a paired, participant-level bootstrap. The comparison is compositional: all six weights are normalized to sum to 100 before deltas are calculated.

A shift means the reported importance changed between sign-up and recall. It may reflect learning, memory, justification, measurement noise, or the change in question wording; the data cannot distinguish these mechanisms.

In [6]:
pref_cols = [c for c in [*STATED, *FOLLOWUP] if c in data.columns]
if not all(c in data.columns for c in [*STATED, *FOLLOWUP]):
    print('One or more preference columns are absent; skip paired preference comparison.')
else:
    repeated_consistency = data.groupby('iid')[[*STATED, *FOLLOWUP]].nunique(dropna=True).gt(1).any(axis=1)
    print(f'Participants with inconsistent repeated preference values across date rows: {repeated_consistency.sum()}')

    people = data.groupby('iid')[[*STATED, *FOLLOWUP, 'gender', 'wave']].first()
    pre_raw = people[STATED].apply(pd.to_numeric, errors='coerce')
    post_raw = people[FOLLOWUP].apply(pd.to_numeric, errors='coerce')

    def normalize_composition(values):
        values = values.where(values.ge(0))
        totals = values.sum(axis=1, min_count=values.shape[1])
        return values.div(totals.where(totals > 0), axis=0) * 100

    pre = normalize_composition(pre_raw)
    post = normalize_composition(post_raw)
    pre.columns = ATTRS
    post.columns = ATTRS
    complete_pair = pre.notna().all(axis=1) & post.notna().all(axis=1)
    pre_pair, post_pair = pre.loc[complete_pair], post.loc[complete_pair]
    delta = post_pair - pre_pair
    n_pair = len(delta)
    print(f'Complete paired participants: {n_pair:,} / {len(people):,}')

    if n_pair:
        rng = np.random.default_rng(SEED)
        delta_values = delta.to_numpy()
        boot_means = np.empty((N_BOOT, len(ATTRS)))
        for b in range(N_BOOT):
            idx = rng.integers(0, n_pair, size=n_pair)
            boot_means[b] = delta_values[idx].mean(axis=0)
        ci = np.quantile(boot_means, [0.025, 0.975], axis=0)
        paired_results = pd.DataFrame({
            'attribute': ATTRS,
            'pre_mean': pre_pair.mean().to_numpy(),
            'followup_mean': post_pair.mean().to_numpy(),
            'mean_change_pp': delta.mean().to_numpy(),
            'bootstrap_95CI_low': ci[0], 'bootstrap_95CI_high': ci[1],
            'share_increased': (delta > 0).mean().to_numpy(),
        })
        display(paired_results.round(2))
        total_variation = 0.5 * delta.abs().sum(axis=1)
        print(f'Mean within-person pre/follow-up reallocation: {total_variation.mean():.1f} points; '
              f'median={total_variation.median():.1f}; 95th percentile={total_variation.quantile(.95):.1f}')

Participants with inconsistent repeated preference values across date rows: 0
Complete paired participants: 123 / 551
  attribute  pre_mean  followup_mean  mean_change_pp  bootstrap_95CI_low  bootstrap_95CI_high  share_increased
0      attr     23.87          32.17            8.30                6.03                10.86             0.62
1      sinc     17.00          14.16           -2.84               -4.28                -1.60             0.22
2     intel     20.11          15.76           -4.35               -5.78                -2.94             0.15
3       fun     18.08          18.38            0.30               -1.38                 1.96             0.37
4       amb      9.24           7.42           -1.82               -2.85                -0.85             0.21
5      shar     11.69          12.11            0.42               -1.21                 1.89             0.43
Mean within-person pre/follow-up reallocation: 19.9 points; median=19.0; 95th percentile=44.5


## 6. Segment participants by the size of their say–do gap

For each participant with enough yes and no decisions, calculate the within-person standardized rating difference between dates they accepted and rejected for each attribute. Positive differences are converted into a six-attribute behavioral importance proxy that sums to 100. Compare that proxy with the participant's normalized pre-event `*1_1` stated weights using total-variation distance:

`gap = 0.5 × sum(abs(stated_weight − behavioral_weight))`

The result ranges from 0 to 100 percentage points. A high gap means more of the 100-point allocation would need to move to make the two profiles agree. Terciles are descriptive segments; they are not discovered latent classes. Because the behavioral proxy is calculated from the same event's decisions and date ratings, do not claim it predicts future matches without a held-out validation sample.

In [7]:
def derive_gap_segments(min_each=2):
    required_segment = ['iid', 'dec', *ATTRS, *STATED]
    d = data.dropna(subset=required_segment).copy()
    for a in ATTRS:
        means = d.groupby('iid')[a].transform('mean')
        sds = d.groupby('iid')[a].transform(lambda s: s.std(ddof=0))
        d[f'z_{a}'] = ((d[a] - means) / sds.replace(0, np.nan)).fillna(0.0)

    out = []
    no_positive_signal = 0
    too_few_choices = 0
    for iid, g in d.groupby('iid', sort=False):
        yes = g.loc[g['dec'].eq(1)]
        no = g.loc[g['dec'].eq(0)]
        if len(yes) < min_each or len(no) < min_each:
            too_few_choices += 1
            continue
        differences = np.array([yes[f'z_{a}'].mean() - no[f'z_{a}'].mean() for a in ATTRS], dtype=float)
        positive = np.clip(differences, 0, None)
        if not np.isfinite(positive).all() or positive.sum() <= 0:
            no_positive_signal += 1
            continue
        behavior = positive / positive.sum() * 100
        stated = pd.to_numeric(g.iloc[0][STATED], errors='coerce').to_numpy(dtype=float)
        if not np.isfinite(stated).all() or stated.sum() <= 0:
            continue
        stated = stated / stated.sum() * 100
        gap = 0.5 * np.abs(stated - behavior).sum()
        row = {
            'iid': iid, 'gender': g['gender'].iloc[0] if 'gender' in g else np.nan,
            'wave': g['wave'].iloc[0] if 'wave' in g else np.nan,
            'n_dates': len(g), 'n_yes': len(yes), 'n_no': len(no),
            'say_do_gap_pp': gap,
            'top_stated_attribute': ATTRS[int(np.argmax(stated))],
            'top_behavior_attribute': ATTRS[int(np.argmax(behavior))],
        }
        if 'match' in g.columns:
            row['mutual_match_rate'] = g['match'].mean()
        for a, value in zip(ATTRS, stated):
            row[f'stated_{a}'] = value
        for a, value in zip(ATTRS, behavior):
            row[f'behavior_{a}'] = value
        out.append(row)

    result = pd.DataFrame(out)
    result.attrs['too_few_choices'] = too_few_choices
    result.attrs['no_positive_signal'] = no_positive_signal
    if result.empty:
        return result
    rank_pct = result['say_do_gap_pp'].rank(method='average', pct=True)
    result['gap_segment'] = pd.cut(
        rank_pct, bins=[0, 1/3, 2/3, 1.0000001],
        labels=['Low gap', 'Medium gap', 'High gap'], include_lowest=True
    )
    return result

segments_2 = derive_gap_segments(min_each=2)
print(f"Participants eligible (>=2 yes and >=2 no): {len(segments_2):,}; "
      f"excluded for too few choices: {segments_2.attrs.get('too_few_choices', 0):,}; "
      f"excluded because no positive behavioral signal: {segments_2.attrs.get('no_positive_signal', 0):,}")
if not segments_2.empty:
    agg = {
        'participants': ('iid', 'nunique'),
        'mean_gap_pp': ('say_do_gap_pp', 'mean'),
        'median_gap_pp': ('say_do_gap_pp', 'median'),
        'top_behavior_attribute': ('top_behavior_attribute', lambda s: s.value_counts().index[0]),
    }
    if 'mutual_match_rate' in segments_2.columns:
        agg['mean_mutual_match_rate'] = ('mutual_match_rate', 'mean')
    segment_summary = segments_2.groupby('gap_segment', observed=False).agg(**agg)
    display(segment_summary.round(3))

    profile_cols = [f'stated_{a}' for a in ATTRS] + [f'behavior_{a}' for a in ATTRS]
    segment_profiles = segments_2.groupby('gap_segment', observed=False)[profile_cols].mean()
    display(segment_profiles.round(1))

# Sensitivity to a stricter minimum number of yes/no observations for each person.
segments_3 = derive_gap_segments(min_each=3)
sensitivity = pd.DataFrame([
    {'minimum_each_outcome': 2, 'participants': len(segments_2),
     'median_gap_pp': segments_2['say_do_gap_pp'].median() if len(segments_2) else np.nan},
    {'minimum_each_outcome': 3, 'participants': len(segments_3),
     'median_gap_pp': segments_3['say_do_gap_pp'].median() if len(segments_3) else np.nan},
])
display(sensitivity.round(2))

Participants eligible (>=2 yes and >=2 no): 398; excluded for too few choices: 129; excluded because no positive behavioral signal: 0
             participants  mean_gap_pp  median_gap_pp top_behavior_attribute  mean_mutual_match_rate
gap_segment                                                                                         
Low gap               132       16.253         16.566                   attr                   0.202
Medium gap            133       28.927         28.845                   attr                   0.164
High gap              133       47.168         44.755                   attr                   0.192
             stated_attr  stated_sinc  stated_intel  stated_fun  stated_amb  stated_shar  behavior_attr  behavior_sinc  behavior_intel  behavior_fun  behavior_amb  behavior_shar
gap_segment                                                                                                                                                                      
Low g

## 7. Out-of-sample ML robustness: do stated preferences add predictive value?

The earlier `03_ml.ipynb` compared logistic regression with a gradient booster on date ratings. This extension asks a different question: **do stated preferences improve prediction of an individual's yes/no decision beyond date ratings?**

We compare four feature sets across two algorithms (regularized logistic regression and gradient boosting): ratings only; ratings plus stated weights; those features plus rating-by-stated-preference interactions; and the interaction model plus date order and choice condition. Each participant is held out together in one split, and a stricter check holds out whole waves so no event appears in both training and test. Scores include ROC AUC, log loss, Brier score, 0.5-threshold accuracy, and the top-three accepted-date rate. Accuracy is compared with the majority-class baseline.

**Limit:** `attr`, `sinc`, etc. are ratings recorded during/after each date. Therefore this evaluates prediction of the decision in this experiment, not a pre-date dating-app recommendation. The condition feature is also event-confounded, as shown above. Peer average ratings are excluded from this ML evaluation because they are post-event and could leak information across folds."

In [8]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ml_required = ['iid', 'wave', 'dec', 'order', 'condtn', *ATTRS, *STATED]
ml_frame = data.dropna(subset=ml_required).copy()
core_people = ml_frame.groupby('iid')['dec'].nunique()
core_people = core_people[core_people.eq(2)].index
ml_frame = ml_frame[ml_frame['iid'].isin(core_people)].copy()

ord_min = ml_frame.groupby('iid')['order'].transform('min')
ord_max = ml_frame.groupby('iid')['order'].transform('max')
ml_frame['order_rel'] = (ml_frame['order'] - ord_min) / (ord_max - ord_min).replace(0, np.nan)
ml_frame = ml_frame.dropna(subset=['order_rel']).copy()

interaction_features = []
for attr, stated_col in zip(ATTRS, STATED):
    col = f'weighted_{attr}'
    ml_frame[col] = ml_frame[attr] * ml_frame[stated_col] / 100.0
    interaction_features.append(col)

feature_sets = {
    'Ratings only': list(ATTRS),
    'Ratings + stated weights': [*ATTRS, *STATED],
    'Plus rating × stated interactions': [*ATTRS, *STATED, *interaction_features],
    'Plus order + condition': [*ATTRS, *STATED, *interaction_features, 'order_rel', 'condtn'],
}

y = ml_frame['dec'].astype(int).to_numpy()
cv_groups = {
    'Participant grouped': ml_frame['iid'].to_numpy(),
    'Wave grouped': ml_frame['wave'].to_numpy(),
}


def make_classifier(model_name):
    if model_name == 'Logistic regression':
        return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, random_state=SEED))
    return GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=2,
        min_samples_leaf=20, subsample=0.8, random_state=SEED
    )

ml_rows = []
ml_oof = {}
for split_name, groups in cv_groups.items():
    splitter = GroupKFold(n_splits=5)
    for model_name in ['Logistic regression', 'Gradient boosting']:
        for feature_name, features in feature_sets.items():
            X = ml_frame[features].astype(float)
            oof = np.full(len(ml_frame), np.nan)
            fold_auc, fold_logloss = [], []
            for train_idx, test_idx in splitter.split(X, y, groups=groups):
                model = make_classifier(model_name)
                model.fit(X.iloc[train_idx], y[train_idx])
                fold_prob = model.predict_proba(X.iloc[test_idx])[:, 1]
                oof[test_idx] = fold_prob
                if np.unique(y[test_idx]).size == 2:
                    fold_auc.append(roc_auc_score(y[test_idx], fold_prob))
                fold_logloss.append(log_loss(y[test_idx], fold_prob, labels=[0, 1]))

            valid = np.isfinite(oof)
            pred = pd.DataFrame({
                'iid': ml_frame.loc[valid, 'iid'].to_numpy(),
                'actual': y[valid], 'probability': oof[valid],
            })
            pred['rank_within_person'] = pred.groupby('iid')['probability'].rank(method='first', ascending=False)
            top3 = pred.loc[pred['rank_within_person'].le(3)].groupby('iid')['actual'].mean().mean()
            ml_rows.append({
                'CV grouping': split_name, 'model': model_name, 'feature set': feature_name,
                'n_decisions': int(valid.sum()), 'n_people': int(ml_frame.loc[valid, 'iid'].nunique()),
                'yes_rate': float(np.mean(y[valid])),
                'OOF AUC': roc_auc_score(y[valid], oof[valid]),
                'mean fold AUC': np.mean(fold_auc), 'fold AUC SD': np.std(fold_auc, ddof=1),
                'OOF log loss': log_loss(y[valid], oof[valid], labels=[0, 1]),
                'OOF Brier': brier_score_loss(y[valid], oof[valid]),
                'accuracy at 0.5': accuracy_score(y[valid], oof[valid] >= 0.5),
                'majority baseline accuracy': max(np.mean(y[valid]), 1 - np.mean(y[valid])),
                'top-3 yes rate': top3,
            })
            ml_oof[(split_name, model_name, feature_name)] = oof

ml_results = pd.DataFrame(ml_rows)
display(ml_results.round(4))

# Incremental gains versus ratings-only, paired on the same held-out rows.
gain_rows = []
for split_name in cv_groups:
    for model_name in ['Logistic regression', 'Gradient boosting']:
        base = ml_results.query("`CV grouping` == @split_name and model == @model_name and `feature set` == 'Ratings only'").iloc[0]
        for feature_name in list(feature_sets)[1:]:
            current = ml_results.query("`CV grouping` == @split_name and model == @model_name and `feature set` == @feature_name").iloc[0]
            gain_rows.append({
                'CV grouping': split_name, 'model': model_name, 'added features': feature_name,
                'delta AUC': current['OOF AUC'] - base['OOF AUC'],
                'delta log loss': current['OOF log loss'] - base['OOF log loss'],
                'delta Brier': current['OOF Brier'] - base['OOF Brier'],
                'delta top-3 yes rate': current['top-3 yes rate'] - base['top-3 yes rate'],
            })
ml_gains = pd.DataFrame(gain_rows)
display(ml_gains.round(4))

            CV grouping                model                        feature set  n_decisions  n_people  yes_rate  OOF AUC  mean fold AUC  fold AUC SD  OOF log loss  OOF Brier  accuracy at 0.5  majority baseline accuracy  top-3 yes rate
0   Participant grouped  Logistic regression                       Ratings only         6222       452    0.4392   0.8277         0.8279       0.0144        0.5078     0.1681           0.7507                      0.5608          0.7799
1   Participant grouped  Logistic regression           Ratings + stated weights         6222       452    0.4392   0.8268         0.8272       0.0146        0.5089     0.1685           0.7486                      0.5608          0.7806
2   Participant grouped  Logistic regression  Plus rating × stated interactions         6222       452    0.4392   0.8263         0.8268       0.0152        0.5093     0.1688           0.7462                      0.5608          0.7872
3   Participant grouped  Logistic regression            

## 8. What to report

Use the tables above to answer the following without overstating the evidence:

1. **Consensus sensitivity:** Do consensus ratings add fit? For which attributes does the participant's own rating relative to consensus retain an association with yes decisions?
2. **Choice architecture:** Is the extensive-choice condition associated with a different yes rate? Is the contrast still estimable within waves that contain both conditions? If not, describe it as an event-level association, not a causal choice-overload effect.
3. **Order:** Does yes propensity rise or fall from first to last date? Does the quadratic term improve fit? Call this an order effect unless fatigue is separately identified.
4. **Retrospective preference:** Which attributes show the largest average pre-event to follow-up shifts, and how large is the typical total reallocation? Describe this as recall/reported-weight change.
5. **Segments:** What do the high-gap participants look like in their stated and behavior-linked profiles? Are mutual-match rates different descriptively? This is a useful targeting hypothesis for a future platform test, not proof that a segment-specific recommendation policy will improve matches.
6. **Predictive robustness:** Do stated weights or their interactions improve out-of-sample AUC, log loss, Brier score, accuracy over the majority baseline, or top-three yes rate? Give more weight to improvements that persist when entire waves are held out.

### Marketing-analytics interpretation

The strongest marketing framing is **preference-based recommendation and choice architecture**: stated preferences, event behavior, and partner consensus may each add different signals to matching. The gap segments suggest a testable onboarding strategy (for example, ask users to allocate preferences, then compare their top recommendations with observed interaction feedback). This dataset does not contain real app recommendations, exposure logs, subscription behavior, retention, or revenue, so it cannot estimate a commercial lift, CLV, or an optimal price plan.

### Method references

- [Speed Dating Data Key](https://web.mit.edu/17.871/www/2009/speed_dating_codebook.pdf) — field definitions and questionnaire timing.
- [Fisman, Iyengar, Kamenica & Simonson (2006)](https://faculty.chicagobooth.edu/emir.kamenica/documents/genderDifferences.pdf) — original study context.
- The core-model notebook uses conditional fixed-effects logit and participant-level bootstrap; this notebook follows its participant grouping for conditional logits but treats cross-partner and event dependence as an important limitation.